# Audio Reconstruction

In this notebook we will use the [Griffin-Lim algorithm](https://ieeexplore.ieee.org/document/1164317) to reconstruct an audio signal from its magnitude spectrogram. 

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import pickle
from tqdm import tqdm
import torch
import torchaudio
import torch.nn as nn
import librosa 
import soundfile as sf


from paths import *
from reshandler import DictResHandler
# from model_dataset import MelSpecTransformDBNoNorm as TheTransform
from model_dataset import DeNormalizerMVNManual

In [3]:
transform_configs = {
    "sample_rate": 16000,
    "n_fft": 512,
    "hop_length": 128,
    "n_mels": 96,  
}

In [4]:
def read_result_at(res_save_dir, epoch): 
    all_handler = DictResHandler(whole_res_dir=res_save_dir, 
                                 file_prefix=f"all-{epoch}")

    all_handler.read()

    return all_handler.res

In [5]:
class MelSpecTransformDBNoNorm(nn.Module): 
    """
    20241113: Added the part to customize hop_length
    """
    def __init__(self, sample_rate, n_fft=400, n_mels=64, hop_length=None): 
        super().__init__()
        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.hop_length = hop_length


        n_stft = int((n_fft//2) + 1)
        if hop_length is None:
            hop_length = int(n_fft//2)
        self.transform = torchaudio.transforms.MelSpectrogram(sample_rate, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length)
        self.inverse_mel = torchaudio.transforms.InverseMelScale(sample_rate=sample_rate, n_mels=n_mels, n_stft=n_stft)
        self.grifflim = torchaudio.transforms.GriffinLim(n_fft=n_fft, hop_length=hop_length)
        self.amplitude_to_DB = torchaudio.transforms.AmplitudeToDB(stype='power')

    def forward(self, waveform): 
        # transform to mel_spectrogram
        mel_spec = self.transform(waveform)  # (channel, n_mels, time)
        # mel_spec = F.amplitude_to_DB(mel_spec)
        mel_spec = self.amplitude_to_DB(mel_spec)
        # mel_spec = torch.tensor(librosa.power_to_db(mel_spec.squeeze().numpy()))
        mel_spec = mel_spec.squeeze()
        mel_spec = mel_spec.permute(1, 0) # (F, L) -> (L, F)
        return mel_spec
    
    def inverse(self, mel_spec): 
        mel_spec = mel_spec.permute(1, 0) # (L, F) -> (F, L)
        # mel_spec = torch.tensor(librosa.db_to_power(mel_spec.numpy()))
        mel_spec = torchaudio.functional.DB_to_amplitude(mel_spec, ref=1.0, power=1)
        mel_spec = mel_spec.unsqueeze(0)  # restore from (F, L) to (channel, F, L)
        i_mel = self.inverse_mel(mel_spec)
        inv = self.grifflim(i_mel)
        return inv
    
    def inverse_librosa(self, mel_spec): 
        # Assuming `mel_spec` is a PyTorch tensor 
        mel_spec = np.transpose(mel_spec, (1, 0))  # (L, F) -> (F, L) and convert to NumPy 
        mel_spec = librosa.db_to_power(mel_spec, ref=1.0)  # Convert dB to amplitude 

        # Perform the mel-to-STFT conversion using Librosa 
        i_mel = librosa.feature.inverse.mel_to_stft(mel_spec, sr=self.sample_rate, n_fft=self.n_fft) 

        # Use Griffin-Lim for phase reconstruction 
        inv = librosa.griffinlim(i_mel, hop_length=self.hop_length, n_iter=32) 
        return inv

In [6]:
train_name = "E_0A"
ts = "1113024340"
eval_dir = os.path.join(model_save_, f"eval-{train_name}-{ts}")

# Load MV_config
with open(os.path.join(src_, "mv_config_sashi_512_128_96.pkl"), "rb") as file: 
    mv_config = pickle.load(file)

# Load Normalizer
denormalizer = DeNormalizerMVNManual()
mytrans = MelSpecTransformDBNoNorm(sample_rate=transform_configs["sample_rate"], 
                    n_fft=transform_configs["n_fft"], n_mels=transform_configs["n_mels"],
                    hop_length=transform_configs["hop_length"])

In [ ]:
dim = 32
balance_type = "b"
run_number = random.randint(1, 5)
target_res_dir = os.path.join(eval_dir, f"recon{dim}-phi", f"{balance_type}", f"{run_number}")
output_dir = os.path.join(eval_dir, f"recon{dim}-phi", f"{balance_type}", "audio_reconstruction")
mk(output_dir)

# random idx to sample
idx = random.randint(0, 900)
outoutput_dir = os.path.join(output_dir, f"run-{run_number}-sample-{idx}")
mk(outoutput_dir)
print(run_number, idx)

for epoch in tqdm(range(101)): # 0~100
    allres = read_result_at(target_res_dir, epoch)
    original_mel = allres["ori"][idx]
    reconstructed_mel = allres["recon"][idx]
    v1_name = allres["v1-name"][idx]
    s_name = allres["sn"][idx]
    v2_name = allres["vn"][idx]

    if epoch == 0: 
        original_mel_denorm = denormalizer(original_mel, mv_config["mean"], mv_config["std"])
        original_audio = mytrans.inverse_librosa(original_mel_denorm)
        sf.write(os.path.join(outoutput_dir, f"{v1_name+s_name+v2_name}-epoch-{epoch:03}-original.flac"), original_audio, transform_configs["sample_rate"])
    # Transform original and reconstructed Mel-spectrograms to audio
    reconstructed_mel_denorm = denormalizer(reconstructed_mel, mv_config["mean"], mv_config["std"])
    reconstructed_audio = mytrans.inverse_librosa(reconstructed_mel_denorm)
    # Save audio
    sf.write(os.path.join(outoutput_dir, f"{v1_name+s_name+v2_name}-epoch-{epoch:03}-reconstructed.flac"), reconstructed_audio, transform_configs["sample_rate"])

3 697


100%|██████████| 101/101 [00:21<00:00,  4.60it/s]


: 